# Figure 1 Jacobian supplement | Supplementary Fig. S1f-j

This notebook validates CNN structural connectivity using input-averaged Jacobians and evaluates Transformer Jacobian reliability across input splits. The MLP Jacobian analysis remains in `Jacobian_analysis.ipynb` (Extended Data Fig. 1e).

## 1. Before running

Export the final `state_dict` once from each main analysis notebook/script:

```python
# CNN notebook
torch.save(model_states[-1], '../Supplementary_fig_code/checkpoints/cnn_final.pt')

# Transformer analysis
torch.save(model_states[-1], '../Supplementary_fig_code/checkpoints/transformer_final.pt')
```

If your checkpoint architecture differs from the defaults below, edit the corresponding model configuration before loading it. For publication runs, use a deterministic validation loader (`shuffle=False`).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

# Work both when Jupyter starts in the project root and in this notebook folder.
candidates = [
    Path.cwd(),
    Path.cwd() / 'FC-IS_code' / 'Supplementary_fig_code',
]
CODE_DIR = next((path.resolve() for path in candidates if (path / 'utils' / 'fig1_jacobian.py').exists()), None)
if CODE_DIR is None:
    raise FileNotFoundError('Could not locate Supplementary_fig_code/utils. Start Jupyter in the project root or notebook folder.')
PROJECT_ROOT = CODE_DIR.parents[1]
sys.path.insert(0, str(CODE_DIR))

from utils.models import SimpleCNN, TransformerLM, load_state_dict_file
from utils.fig1_jacobian import (
    SuppFig1Config,
    assemble_results,
    load_results,
    plot_fig1_jacobian_supp,
    run_cnn_validation,
    run_transformer_validation,
    save_results,
    set_seed,
)

print('Code directory:', CODE_DIR)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2. Configuration

The default architecture values reproduce the current main-analysis code: two 100-unit MLP hidden layers, the 32/64-channel CNN, and a two-layer Transformer with `d_model=128`. The CNN Jacobian is evaluated only for a reproducible subset of output units because a full 1,600 × 6,272 Jacobian is unnecessarily expensive for this validation.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT_DIR = CODE_DIR / 'checkpoints'
RESULT_DIR = CODE_DIR / 'results' / 'fig1_Jacobian_supp'
OUTPUT_DIR = CODE_DIR / 'outputs' / 'fig1_Jacobian_supp'
for folder in (CHECKPOINT_DIR, RESULT_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CNN_CHECKPOINT = CHECKPOINT_DIR / 'cnn_final.pt'
TRANSFORMER_CHECKPOINT = CHECKPOINT_DIR / 'transformer_final.pt'
RESULT_CACHE = RESULT_DIR / 'fig1_Jacobian_supp_source_data.npz'

CNN_DATA_ROOT = CODE_DIR.parent / 'CNN' / 'data'
TRANSFORMER_DATA_DIR = CODE_DIR.parent / 'Transformer' / 'data' / 'wikitext-2'
DOWNLOAD_IMAGE_DATA = False  # Set True only if the local torchvision data are absent.

config = SuppFig1Config(
    seed=42,
    cnn_n_samples=64,
    cnn_n_output_units=64,
    transformer_layer_index=0,
    transformer_n_samples=64,
    transformer_chunk_size=2,  # Reduce to 1 if GPU memory is limited.
    transformer_repeats=100,
    transformer_sample_sizes=(1, 2, 4, 8, 16, 32),
)
set_seed(config.seed)
print(config)

## 3. Deterministic validation data loaders

Use the same normalization and tokenization as the main code. If your Transformer checkpoint used another vocabulary file or tokenizer, replace only `build_transformer_loader`; the SC validation functions do not depend on a particular text dataset implementation.

In [ ]:
from collections import Counter
from torchvision import datasets, transforms

def build_cnn_loader(batch_size=64):
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    dataset = datasets.MNIST(
        root=CNN_DATA_ROOT, train=False, transform=transform, download=DOWNLOAD_IMAGE_DATA
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

class WordSequenceDataset(Dataset):
    def __init__(self, text, vocab, seq_len=32):
        token_ids = [vocab.get(word, 1) for word in text.split()]
        total = (len(token_ids) // seq_len) * seq_len
        self.data = torch.tensor(token_ids[:total], dtype=torch.long).view(-1, seq_len)

    def __len__(self):
        return max(len(self.data) - 1, 0)

    def __getitem__(self, index):
        return self.data[index], self.data[index + 1]

def build_transformer_loader(batch_size=16, vocab_size=5000, seq_len=32):
    train_path = TRANSFORMER_DATA_DIR / 'wiki.train.tokens'
    validation_path = TRANSFORMER_DATA_DIR / 'wiki.valid.tokens'
    if not train_path.exists() or not validation_path.exists():
        raise FileNotFoundError(
            'WikiText-2 files are missing. Set TRANSFORMER_DATA_DIR to the same local data used in the main analysis.'
        )
    train_text = train_path.read_text(encoding='utf-8')
    validation_text = validation_path.read_text(encoding='utf-8')
    most_common = [word for word, _ in Counter(train_text.split()).most_common(vocab_size - 2)]
    vocab = {'<pad>': 0, '<unk>': 1}
    vocab.update({word: index + 2 for index, word in enumerate(most_common)})
    dataset = WordSequenceDataset(validation_text, vocab, seq_len=seq_len)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0), vocab

## 4. Load trained models

Checkpoint loading is strict by default. A key mismatch usually means the configuration below does not match the model that produced the checkpoint; fix the configuration instead of using `strict=False` without inspecting the mismatch.

In [ ]:
required_checkpoints = [CNN_CHECKPOINT, TRANSFORMER_CHECKPOINT]
missing = [str(path) for path in required_checkpoints if not path.exists()]
if missing:
    raise FileNotFoundError('Export the trained state_dict files described in Section 1. Missing:\n' + '\n'.join(missing))

cnn_model = SimpleCNN(in_channels=1, activation='relu')
transformer_model = TransformerLM(
    vocab_size=5000, d_model=128, n_heads=4, d_ff=256,
    n_layers=2, seq_len=32, dropout=0.1,
)

load_state_dict_file(cnn_model, str(CNN_CHECKPOINT), map_location='cpu')
load_state_dict_file(transformer_model, str(TRANSFORMER_CHECKPOINT), map_location='cpu')

cnn_loader = build_cnn_loader()
transformer_loader, transformer_vocab = build_transformer_loader()
print('Models and deterministic validation loaders are ready.')

## 5. Compute CNN Jacobian validation

In [ ]:
cnn_results = run_cnn_validation(cnn_model, cnn_loader, config, device=DEVICE)
print({key: value.shape for key, value in cnn_results.items()})

## 6. Compute Transformer input-split reliability

In [ ]:
transformer_results = run_transformer_validation(
    transformer_model, transformer_loader, config, device=DEVICE
)
print({key: value.shape for key, value in transformer_results.items()})

## 7. Save source data

In [ ]:
results = assemble_results(
    cnn_results,
    transformer_results,
    config,
    metadata={
        'cnn_checkpoint': CNN_CHECKPOINT.name,
        'transformer_checkpoint': TRANSFORMER_CHECKPOINT.name,
        'device': DEVICE,
    },
)
save_results(results, RESULT_CACHE)
print('Saved source data:', RESULT_CACHE)

## 8. Draw and export Supplementary Fig. S1f-j

In [ ]:
# Safe to replace the previous line with: results = load_results(RESULT_CACHE)
figure, statistics = plot_fig1_jacobian_supp(
    results,
    output_prefix=OUTPUT_DIR / 'fig1_Jacobian_supp',
    export_formats=['svg', 'pdf', 'tiff', 'png'],
    dpi=600,
)
display(figure)
statistics

## Reporting checklist

- f-g: CNN kernel-average SC versus expected Jacobian and admissible-edge agreement.
- h-j: Transformer split-half Jacobian SC and sampling convergence.
- MLP Jacobian is intentionally excluded and remains in `Jacobian_analysis.ipynb` (Fig. S1e).